# [Chapter 11 — Model Generalizations, §11.2] The $SIRS_I$ Model: Loss of Immunity and Endemic Equilibrium

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 11 — Model Generalizations
**Considerations developed:** 13 (equilibrium vs invasion conflation)
**Estimated runtime:** ≤ 20 s

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/chapter_11_generalizations/02_SIRS_loss_of_immunity.ipynb)


## What this notebook does

Adds a return flow $R \to S$ at rate $\xi$ to the basic $SIR_I$ model. The result is qualitatively different long-term behavior: instead of a single epidemic that burns through the population and ends, the system approaches an endemic equilibrium where new infections are continuously balanced by recoveries and immune waning. The notebook makes Consideration~13 concrete by computing both the invasion-phase $\mathcal{R}_0$ and the steady-state $I^*$, and showing that they are *different questions* about the same model.

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from shared import book_style, BOOK_COLORS
from shared.seeds import set_seed_chapter_11
from shared.verification import assert_within_tolerance
set_seed_chapter_11()
book_style()


## $SIRS_I$ system

In [ ]:
from shared.parameters import baseline_chapter_11
p = baseline_chapter_11()
p['t_end'] = 365.0 * 5  # show 5 years
p['n_points'] = 5001

def sirs_rhs(y, t, p):
    S, I, R = y
    N = p['N']
    inc = p['c_I'] * p['beta'] * S * I / N
    return [-inc + p['xi']*R,
            inc - I/p['tau_R'],
            I/p['tau_R'] - p['xi']*R]

y0 = [p['S0'], p['I0'], p['R0_init']]
t = np.linspace(p['t_start'], p['t_end'], p['n_points'])
sol = odeint(sirs_rhs, y0, t, args=(p,))
S, I, R = sol.T

R0 = p['c_I']*p['beta']*p['tau_R']
# Endemic equilibrium (closed form for SIRS): S* = N/R_0; I* = (mu_loss N)(1 - 1/R_0) / (1/tau_R + mu_loss)
# Here we have no demographic births so xi acts as the only return flow:
I_star = (p['xi'] * (1 - 1.0/R0)) / (p['xi'] + 1.0/p['tau_R'])
print(f'R_0 (invasion) = {R0:.3f}')
print(f'I* (endemic eq.) = {I_star:.5f}')


### Trajectory: epidemic followed by endemicity

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(t/365.0, I, color=BOOK_COLORS['infectious'], lw=2, label='$I(t)$')
ax.axhline(I_star, ls='--', color=BOOK_COLORS['highlight'], label='endemic $I^*$')
ax.set_xlabel('time (years)')
ax.set_ylabel('infectious fraction')
ax.set_title('$SIRS_I$: invasion + endemic equilibrium')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


### Why $\mathcal{R}_0$ alone does not predict $I^*$ (Consideration~13)

Two diseases with the *same* $\mathcal{R}_0$ but different $\xi$ produce vastly different endemic burdens. Reporting $\mathcal{R}_0$ without $I^*$ communicates *whether* the disease invades but not *how heavy* the long-term burden will be.

In [ ]:
results = []
for xi_days in [180, 365, 365*3]:
    xi = 1.0/xi_days
    Ist = (xi * (1 - 1.0/R0)) / (xi + 1.0/p['tau_R'])
    results.append((xi_days, R0, Ist))
print(f"{'mean immunity (d)':>22} | {'R_0':>5} | {'I*':>8}")
for xi_d, r, Ist in results:
    print(f'{xi_d:>22} | {r:>5.2f} | {Ist:>8.5f}')

# Verification: I* must be monotonically increasing in xi (faster waning -> higher endemic burden)
I_stars = [r[2] for r in results]
assert I_stars[0] > I_stars[1] > I_stars[2], 'I* should increase as immunity duration shortens'
print('\nVerified: I* increases monotonically as immunity wanes faster.')


## What's next

- The $SEIR_I$ notebook adds latent-period structure orthogonal to immunity dynamics.
- Chapter 12 generalizes from one homogeneous group to multiple interacting subpopulations.
- Chapter 16 endemic-burden discussion ties this back to Consideration~13 in a more general setting.